# Power Laws

*Power laws* are types of distributions where **rare events** are much **more common** compared to the normal distribution.

In [1]:
import numpy as np

from utils.graph import plot_log_log, plot_power_law

## Probability Density Function

$$p(x) = k \cdot x^{-\alpha}, \qquad x \geq x_{\min} > 0, \quad \alpha > 1$$

In [2]:
def p_x(alpha: float, k: float, x: float) -> float:
    return k * (x ** (-alpha))

## Normalization Constant

A PDF must sume to $1$

$$\int_{x_{\min}}^{\infty} p(x)\,dx = 1$$

### Constrains

The integral $\int_0^\infty x^{-\alpha}\,dx$ diverges at $0$ -> lower bound $x_{\min} > 0$.

And also we need $\alpha > 1$ for the tail to converge. For $\alpha < 1$ the density would grow indefinitively.

### Computation

**Step 1: integrate**

$$1 = \int_{x_{\min}}^{\infty} k \cdot x^{-\alpha}\,dx = k \left[ \frac{x^{1-\alpha}}{1-\alpha} \right]_{x_{\min}}^{\infty} = \frac{k}{1-\alpha}\left( \lim_{x \to \infty} x^{1-\alpha} - x_{\min}^{1-\alpha} \right), \qquad \alpha \neq 1$$


**Step 2: evaluate given:** $\alpha > 1$, $\lim_{x \to \infty} x^{1-\alpha} = 0$:

$$1 = \frac{k}{1-\alpha}\left( 0 - x_{\min}^{1-\alpha} \right) = \frac{k}{\alpha-1}\, x_{\min}^{1-\alpha}, \qquad \alpha \gt 1$$

**Result**

$$k = (\alpha-1)\, x_{\min}^{\alpha-1} \qquad\Longrightarrow\qquad p(x) = \frac{\alpha-1}{x_{\min}} \left( \frac{x}{x_{\min}} \right)^{-\alpha}$$

In [3]:
def k_norm(alpha: float, x_min: float) -> float:
    return (alpha - 1) * (x_min ** (alpha - 1))

## Cumulative Distribution Function

$$F(x) = P(X \leq x)$$

**Step 1: write the CDF as an integral of the normalized PDF**

$$F(x) = \int_{x_{\min}}^{x} p(t)\,dt = \int_{x_{\min}}^{x} \frac{\alpha-1}{x_{\min}} \left( \frac{t}{x_{\min}} \right)^{-\alpha} dt$$

**Step 2: make equivalent changes**

$$F(x) = \frac{\alpha-1}{x_{\min}}\, x_{\min}^{\alpha} \int_{x_{\min}}^{x} t^{-\alpha}\,dt = (\alpha-1)\, x_{\min}^{\alpha-1} \int_{x_{\min}}^{x} t^{-\alpha}\,dt$$

**Step 3: integrate**

$$F(x) = (\alpha-1)\, x_{\min}^{\alpha-1} \left[ \frac{t^{1-\alpha}}{1-\alpha} \right]_{x_{\min}}^{x} = -x_{\min}^{\alpha-1}\left( x^{1-\alpha} - x_{\min}^{1-\alpha} \right) = 1 - x_{\min}^{\alpha-1}\, x^{1-\alpha}$$

**Result**

$$F(x) = 1 - \left( \frac{x}{x_{\min}} \right)^{-(\alpha-1)}, \qquad x \geq x_{\min}$$

In [4]:
def cdf_x(alpha: float, x_min: float, x: float) -> float:
    return 1 - (x / x_min) ** (-(alpha - 1))

## Share of the Total Sum

We need a fraction of the **total sum** contributed by observations above a threshold $x$.

**Step 1: contribution to the mean from values above $x$**

$$
M(x)
= \int_x^\infty t\,p(t)\,dt
= k\int_x^\infty t^{1-\alpha}\,dt
= \frac{k}{\alpha-2}x^{2-\alpha},
\qquad \alpha>2
$$

**Step 2: fraction of the total sum contributed by values above $x$**

$$
S(x)
= \frac{M(x)}{M(x_{\min})}
= \left(\frac{x}{x_{\min}}\right)^{-(\alpha-2)}
$$

**Step 3: tail share of the top fraction $q$ of events**

*3a: find the threshold $x_q$*, $q$ determines the amount of elemnts considered from the tail  

$$
1 - F(x_q) = q
$$

$$
\left(\frac{x_q}{x_{\min}}\right)^{-(\alpha-1)} = q
\qquad\Longrightarrow\qquad
\frac{x_q}{x_{\min}} = q^{-\frac{1}{\alpha-1}}
\qquad\Longrightarrow\qquad
x_q = x_{\min}\, q^{-\frac{1}{\alpha-1}}
$$

*3b: put $x_q$ into $S(x)$*

$$
S(x_q)
= \left(\frac{x_q}{x_{\min}}\right)^{-(\alpha-2)}
= \left(q^{-\frac{1}{\alpha-1}}\right)^{-(\alpha-2)}
= q^{\frac{\alpha-2}{\alpha-1}}
$$

In [5]:
def mean_above(alpha: float, k: float, x: float) -> float:
    # Step 1: M(x) = k / (alpha - 2) * x^(2 - alpha)
    return k / (alpha - 2) * x ** (2 - alpha)

def sum_share_above(alpha: float, x_min: float, x: float) -> float:
    # Step 2: S(x) = M(x) / M(x_min)
    k = k_norm(alpha, x_min)
    return mean_above(alpha, k, x) / mean_above(alpha, k, x_min)


def x_top(alpha: float, x_min: float, q: float) -> float:
    # Step 3a: threshold x_q
    q = np.asarray(q, dtype=float)
    with np.errstate(divide="ignore"):  # q = 0 -> x_q = inf
        return x_min * q ** (-1 / (alpha - 1))


def top_share(alpha: float, q: float, x_min: float = 1.0) -> float:
    if alpha <= 2:
        # the mean diverges
        return np.ones_like(np.asarray(q, dtype=float))
    # Step 3b: S(x_q) = q^((alpha - 2) / (alpha - 1))
    return sum_share_above(alpha, x_min, x_top(alpha, x_min, q))

In [6]:
alpha, x_min = 2.16, 1.0
k = k_norm(alpha, x_min)

plot_power_law(
    alpha,
    x_min,
    k,
    pdf=lambda x: p_x(alpha, k, x),
    cdf=lambda x: cdf_x(alpha, x_min, x),
    share=lambda q: top_share(alpha, q, x_min),
)

## Recognizing a Power Law on a Log-Log Plot

**Step 1: take the logarithm of the PDF**

$$\log p(x) = \log\left(k \cdot x^{-\alpha}\right) = \log k - \alpha \log x$$

**Result**

$$Y = \log k - \alpha X \qquad X = \log x \text{ and } Y = \log p(x)$$

In [7]:
def log_p_x(alpha: float, k: float, X: float) -> float:
    # Y = log k - alpha * X, with X = log10(x)
    return np.log10(k) - alpha * X

In [8]:
def sample_power_law(alpha: float, x_min: float, n: int, seed: int = 0) -> np.ndarray:
    # inverse CDF: x = x_min * q^(-1 / (alpha - 1)), q ~ U(0, 1]
    q = 1 - np.random.default_rng(seed).random(n)
    return x_top(alpha, x_min, q)

In [9]:
plot_log_log(
    alpha,
    x_min,
    log_pdf=lambda X: log_p_x(alpha, k, X),
    samples=sample_power_law(alpha, x_min, 100_000),
)